# BigQuery: Conversational Analytics (March 2026 Suite)

[![Open In Colab](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_conversational_analytics_demo.ipynb)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_conversational_analytics_demo.ipynb)

This notebook shows how to use BigQuery's new conversational analytics features — multi-modal queries with `ObjectRef`, built-in AI functions for forecasting and anomaly detection, and job labels for cost tracking.

## Use Case
A data analyst at a retail company needs to investigate sales trends and verify anomalies. They ask a conversational agent to:
1.  **Forecast Sales**: Predict the next 30 days using `AI.FORECAST`.
2.  **Detect Anomalies**: Spot unusual spikes using `AI.DETECT_ANOMALIES`.
3.  **Visual Audit**: Cross-reference a sale with a receipt image in Cloud Storage using `ObjectRef`.

### Release Notes
- [BigQuery Release Notes (March 2026)](https://cloud.google.com/bigquery/docs/release-notes) — `ObjectRef` for GCS integration, `AI.FORECAST` and `AI.DETECT_ANOMALIES` AI functions, job labels for cost attribution
- [ADK v1.28.0](https://github.com/google/adk-python/releases/tag/v1.28.0) — BigQuery Toolset used as the agent interface for this demo

### Requirements
- `google-adk >= 1.28.0` and `google-genai >= 1.69.0` installed.
- BigQuery API and Cloud Storage API enabled.
- A Google Cloud project with billing enabled.

In [ ]:
# 1. Setup and Authentication
%pip install "google-adk>=1.28.0" google-genai google-cloud-bigquery google-cloud-storage nest-asyncio --quiet --index-url https://pypi.org/simple

try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab')
except ModuleNotFoundError:
    print('Not running in Colab — using Application Default Credentials (ADC)')

import os
import nest_asyncio
nest_asyncio.apply()

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
location = 'US' # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = location
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

### 2. [MANDATORY] Project Configuration

Enable the required APIs and set your project.

In [ ]:
# Enable services
!gcloud services enable bigquery.googleapis.com storage.googleapis.com aiplatform.googleapis.com --project={project_id}

# Set project labels for agent activity monitoring (March 2026 Standard)
!gcloud config set project {project_id}
print("Success: Project configured and APIs enabled.")

### 3. [PREREQUISITES] Infrastructure Setup

Creates a partitioned sales table and a GCS bucket with a receipt image for the `ObjectRef` demo.

In [ ]:
from google.cloud import bigquery, storage
import pandas as pd
from datetime import datetime, timedelta

def setup_infrastructure():
    bq_client = bigquery.Client(project=project_id, location=location)
    storage_client = storage.Client(project=project_id)
    
    dataset_id = f"{project_id}.march_demo"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = location
    bq_client.create_dataset(dataset, exists_ok=True)

    # 1. Create Partitioned Table (delete first to avoid duplicate data on re-run)
    table_id = f"{dataset_id}.sales_data"
    schema = [
        bigquery.SchemaField("sale_date", "DATE"),
        bigquery.SchemaField("product_id", "STRING"),
        bigquery.SchemaField("amount", "FLOAT"),
        bigquery.SchemaField("receipt_id", "STRING"),
    ]
    bq_client.delete_table(table_id, not_found_ok=True)
    table = bigquery.Table(table_id, schema=schema)
    table.time_partitioning = bigquery.TimePartitioning(field="sale_date")
    bq_client.create_table(table)

    # 2. Insert Dummy Sales Data
    data = [
        {"sale_date": (datetime.now() - timedelta(days=i)).date().isoformat(), "product_id": f"PROD_{i%5}", "amount": 100.0 + (i*1.5), "receipt_id": f"R_{1000+i}"} 
        for i in range(100)
    ]
    data.append({"sale_date": datetime.now().date().isoformat(), "product_id": "PROD_999", "amount": 15000.0, "receipt_id": "R_ANOMALY"})
    
    bq_client.insert_rows_json(table_id, data)

    # 3. Create GCS Bucket and Dummy Receipt for ObjectRef
    bucket_name = f"{project_id}-receipts"
    bucket = storage_client.create_bucket(bucket_name, location=location) if not storage_client.lookup_bucket(bucket_name) else storage_client.get_bucket(bucket_name)
    
    blob = bucket.blob("receipts/R_ANOMALY.jpg")
    blob.upload_from_string(b"Dummy Receipt Image Data", content_type="image/jpeg")

    print(f"Demo resources prepared: Table '{table_id}' and Bucket 'gs://{bucket_name}'")

setup_infrastructure()

### 4. Core Feature: Conversational Analytics (Multi-Modal & BQML)

BigQuery now supports `ObjectRef` (Cloud Storage references) and BQML AI functions directly through natural language prompts.

We use the ADK BigQuery Toolset to run a conversational agent against the demo data.

In [ ]:
from google.adk import Agent, Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.adk.tools.bigquery import BigQueryToolset
from google.adk.tools.bigquery.config import BigQueryToolConfig
from google.genai import types

# 1. Initialize the BQ Toolset with job labels for cost attribution
#    (captures GOOGLE_CLOUD_LOCATION='US' from env for BQ operations)
bq_config = BigQueryToolConfig(
    job_labels={"ca-bq-job": "true", "demo": "march-2026-suite"}
)
bq_toolset = BigQueryToolset(bigquery_tool_config=bq_config)

# 2. Gemini 3.1 Pro Preview is only available in the 'global' location.
#    Switch env var AFTER BigQueryToolset init so BQ keeps using 'US'.
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

# 3. Define the Conversational Analyst Agent
agent = Agent(
    model="gemini-3.1-pro-preview",
    name="ConversationalAnalyst",
    instruction="""
    You are a senior BigQuery analyst. 
    Use your tools to query partitioned tables efficiently. 
    When asked to forecast or detect anomalies, use BQML AI functions (AI.FORECAST, AI.DETECT_ANOMALIES).
    When asked about receipt images, use the ObjectRef function to reference GCS objects in your SQL.
    Always optimize for cost by using date range filters on the 'sale_date' column.
    """,
    tools=[bq_toolset]
)

# 4. Initialize Runner
runner = Runner(
    agent=agent,
    session_service=InMemorySessionService(),
    app_name="conversational_analytics_demo",
    auto_create_session=True
)

async def perform_conversational_audit():
    prompts = [
        ("Task 1: Forecasting Sales", "Forecast the total daily sales for the next 7 days based on the march_demo.sales_data table using AI.FORECAST."),
        ("Task 2: Detecting Anomalies", "Identify any anomalies in sales amounts for the last 24 hours in march_demo.sales_data using AI.DETECT_ANOMALIES."),
        ("Task 3: Visual Audit (ObjectRef)", f"Show me the receipt image link for the anomalous sale with receipt_id 'R_ANOMALY' using ObjectRef from the gs://{project_id}-receipts bucket."),
    ]
    
    for label, prompt in prompts:
        print(f"\n--- {label} ---")
        print(f"User: {prompt}\n")
        
        message = types.Content(parts=[types.Part(text=prompt)], role='user')
        async for event in runner.run_async(
            user_id="partner_user",
            session_id="march_session",
            new_message=message
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        print(f"Agent: {part.text}")
                    if part.function_call:
                        print(f"[SYSTEM]: Calling tool '{part.function_call.name}'")

await perform_conversational_audit()

### 5. Things to remember or know
- **Multi-modal queries**: `ObjectRef` lets you reference Cloud Storage files (images, PDFs) directly in BigQuery SQL, so you can analyze structured data and unstructured files together.
- **Built-in AI functions**: `AI.FORECAST` and `AI.DETECT_ANOMALIES` work through natural language — no need to train BQML models manually for common tasks.
- **Partition-aware**: The agent automatically adds date-range filters on partitioned tables to reduce cost and improve performance.
- **Job labels**: Agent-initiated jobs are labeled (`ca-bq-job: true`), making it easy to track and attribute costs from AI-driven queries.
- **Runner pattern**: All March 2026 demos use the `Runner` for automatic session management and event streaming.